In [ ]:
import duckdb 
con = duckdb.connect('funds_warehouse.duckdb')

In [ ]:
# Viewing dim_date
print(con.execute ("""
                   DESCRIBE main.dim_date """).fetchdf())

In [ ]:
# Viewing dim_fund
print(con.execute ("""
                   DESCRIBE main.dim_fund """).fetchdf())

In [ ]:
# Viewing fact_etf_prices

print(con.execute ("""
                   DESCRIBE main.fact_etf_prices """).fetchdf())

In [ ]:
# QUERY 1 - Top 10 ETFs by Average Daily Trading Volume
print(con.execute(""" 
                  SELECT 
                        f.fund_symbol, 
                        d.exchange_name, 
                        d.fund_category, 
                        d.fund_family, 
                        ROUND(AVG(f.volume), 0) AS avg_daily_volume, 
                        COUNT(f.price_date) AS trading_days_tracked 
                  FROM main.fact_etf_prices f 
                  LEFT JOIN main.dim_fund d ON f.fund_symbol = d.fund_symbol 
                  GROUP BY f.fund_symbol, d.exchange_name, d.fund_category, d.fund_family 
                  ORDER BY avg_daily_volume DESC LIMIT 10 """).fetchdf())

In [ ]:
# QUERY 2 - Monthly Average Adjusted Close Price
print(con.execute(""" 
                  SELECT 
                        dt.year, 
                        dt.month_name, 
                        dt.month, 
                        ROUND(AVG(f.adj_close), 4) AS avg_adj_close, 
                        ROUND(AVG(f.volume), 0) AS avg_volume, 
                        COUNT(DISTINCT f.fund_symbol) AS active_etfs 
                  FROM main.fact_etf_prices f 
                  JOIN main.dim_date dt ON f.date_key = dt.date_key 
                  GROUP BY dt.year, dt.month, dt.month_name 
                  ORDER BY dt.year, dt.month LIMIT 20 """).fetchdf())

In [ ]:
# QUERY 3 - Most Volatile ETFs by Daily Price Range
print(con.execute(""" 
                  SELECT 
                        f.fund_symbol, 
                        d.fund_category, 
                        ROUND(AVG(f.daily_range), 3) AS avg_daily_range, 
                        ROUND(AVG(f.close), 3) AS avg_close_price, 
                        ROUND(AVG(f.daily_range) / NULLIF(AVG(f.close), 0) * 100, 2) AS volatility_pct 
                  FROM main.fact_etf_prices f 
                  LEFT JOIN main.dim_fund d ON f.fund_symbol = d.fund_symbol 
                  WHERE f.fund_symbol IS NOT NULL 
                  GROUP BY f.fund_symbol, d.fund_category 
                  HAVING count(f.price_date) > 100 
                  ORDER BY volatility_pct DESC LIMIT 15 """).fetchdf())

In [ ]:
# QUERY 4 - SPY Price History (Last 10 Trading Days)
print(con.execute(""" 
                  SELECT 
                        f.price_date, 
                        dt.year, 
                        dt.month_name, 
                        f.open, 
                        f.high, 
                        f.low, 
                        f.close, 
                        f.adj_close, 
                        f.volume, 
                        ROUND(f.daily_return_pct, 4) AS daily_return_pct 
                  FROM main.fact_etf_prices f 
                  JOIN main.dim_date dt ON f.date_key = dt.date_key 
                  WHERE f.fund_symbol = 'SPY' 
                  ORDER BY f.price_date DESC LIMIT 10 """).fetchdf())

In [ ]:
# Close Connection
con.close()